[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FutoshiNakamura-Tts/gaussian-splatting-colab/blob/colab-t4-2025-12-18/gaussian_splatting_colab.ipynb)

In [ ]:
# @title 1. Key Imports & Utilities
import os
import sys
import shutil
import subprocess
import time
import re
import shlex
import torch
import glob
from google.colab import drive, files
from threading import Timer
from queue import Queue
from random import randint
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript

# Output Area for Logs
out = widgets.Output()

def log(msg):
    with out:
        print(msg)

def run_command(cmd, shell=True, env=None):
    try:
        process = subprocess.Popen(
            cmd, shell=shell, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True,
            env=env
        )
        for line in process.stdout:
            log(line.strip())
        process.wait()
        if process.returncode != 0:
             log(f"Command failed with return code {process.returncode}")
    except Exception as e:
        log(f"Error executing command: {e}")


In [ ]:
# @title 2. Configuration
# ===========================
# Configuration Parameters
# ===========================
# @markdown ### Global Config
# (Updated: 2025-12-23 10:07:25)
Mount_Drive = False # @param {type:"boolean"}
Use_Tailscale = False # @param {type:"boolean"}

# @markdown ### Data Config
DataSource = "Demo Data" # @param ["Demo Data", "Google Drive", "Upload Zip", "Custom URL", "Local Folder"]
DrivePath = "/content/drive/MyDrive/my_data.zip" # @param {type:"string"}
Custom_URL = "" # @param {type:"string"}

Repo_Branch = "" # @param {type:"string"}

# @markdown ### Training Config
Source_Path = "/content/tandt/truck" # @param {type:"string"}
Output_Path = "/content/gaussian-splatting/output/my-experiment" # @param {type:"string"}
Iterations = 30000 # @param {type:"integer"}
SH_Degree = 3 # @param {type:"integer"}
White_Background = False # @param {type:"boolean"}
Eval_Mode = False # @param {type:"boolean"}
DryRun = False # @param {type:"boolean"}


In [ ]:
# @title 3. Define Actions
# ===========================
# Action Functions
# ===========================

def setup_env(b):
    out.clear_output()
    log("=== 1. Setting up Environment ===")
    
    if Mount_Drive:
        if not os.path.exists('/content/drive'):
            log("Mounting Google Drive...")
            drive.mount('/content/drive')
        else:
            log("Drive already mounted.")
    
    log("Checking GPU...")
    run_command("nvidia-smi")
    
    log("Environment setup complete.")


def install_deps(b):
    log("\n=== 2. Installing Dependencies ===")
    os.chdir('/content')
    
    # Check Rasterizer
    rasterizer_branch = 'main'
    is_accelerated = False 
    if 'dd_rasterizer' in globals():
        if dd_rasterizer.value.startswith('Accelerated'):
            rasterizer_branch = '3dgs_accel'
            is_accelerated = True
            log("Selected Accelerated Rasterizer (Sparse Adam).")
        else:
            log("Selected Standard Rasterizer.")

    if not os.path.exists('/content/wheels_repo'):
         log("Cloning wheels repo...")
         run_command("git clone -b colab-t4-2025-12-18 https://github.com/FutoshiNakamura-Tts/gaussian-splatting-colab /content/wheels_repo")
    
    repo_dir = '/content/gaussian-splatting'
    if not os.path.exists(repo_dir):
         log(f"Cloning gaussian-splatting...")
         run_command(f"git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting {repo_dir}")
    else:
         pass
    
    log("Installing plyfile...")
    run_command("pip install -q plyfile")

    # Handle Submodules (Diff-Gaussian-Rasterization)
    dgr_dir = f"{repo_dir}/submodules/diff-gaussian-rasterization"
    sknn_dir = f"{repo_dir}/submodules/simple-knn"
    
    try:
        current_dgr_branch = "unknown"
        log(f"Configuring diff-gaussian-rasterization to branch: {rasterizer_branch}")
        
        # Unconditionally fetch and checkout desired branch for submodule
        run_command(f"git -C {dgr_dir} fetch origin {rasterizer_branch}")
        run_command(f"git -C {dgr_dir} checkout {rasterizer_branch}")
    except Exception as e:
        log(f"Error configuring submodule: {e}")

    log("Checking for bundled wheels or building submodules...")
    
    packages = [
        {"name": "diff-gaussian-rasterization", "pattern": "diff_gaussian_rasterization-*-cp*-*-linux_x86_64.whl", "src": dgr_dir},
        {"name": "simple-knn", "pattern": "simple_knn-*-cp*-*-linux_x86_64.whl", "src": sknn_dir}
    ]
    
    for pkg in packages:
        wheel_to_install = None
        
        # 1. If Accelerated, check subfolder first
        if is_accelerated:
            accel_wheels = glob.glob(f"/content/wheels_repo/wheels/3dgs_accel/{pkg['pattern']}")
            if accel_wheels:
                wheel_to_install = accel_wheels[0]
                log(f"Found Accelerated wheel for {pkg['name']}: {wheel_to_install}")
        
        # 2. Check root wheels folder (Fallback or Standard)
        if not wheel_to_install:
             standard_wheels = glob.glob(f"/content/wheels_repo/wheels/{pkg['pattern']}")
             if standard_wheels:
                 wheel_to_install = standard_wheels[0]
                 log(f"Found Standard/Shared wheel for {pkg['name']}: {wheel_to_install}")

        if wheel_to_install:
            run_command(f"{sys.executable} -m pip install --force-reinstall -q {wheel_to_install}")
        else:
            log(f"No wheel found for {pkg['name']}. Building from source...")
            run_command(f"{sys.executable} -m pip install --force-reinstall -q {pkg['src']}")

    log("Dependencies installed.")


def prepare_data(b):
    # Check widgets
    data_source_val = DataSource
    if 'dd_datasource' in globals():
        data_source_val = dd_datasource.value
        
    drive_path_val = DrivePath
    custom_url_val = Custom_URL
    local_folder_val = ""
    
    if 'txt_source_path' in globals():
        # Use the text input for generic path/url
        drive_path_val = txt_source_path.value
        custom_url_val = txt_source_path.value
        local_folder_val = txt_source_path.value

    log(f"\n=== 3. Preparing Data: {data_source_val} ===")
    os.chdir('/content')
    
    if data_source_val == "Demo Data":
        if not os.path.exists('tandt_db'):
            log("Downloading TandT Demo Data...")
            run_command("wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip")
            run_command("unzip -q tandt_db.zip")
        else:
            log("Demo data already exists.")

    elif data_source_val == "Google Drive":
        if not os.path.exists(drive_path_val):
            log(f"Error: Drive Path {drive_path_val} does not exist. Did you mount drive in Step 1?")
            return
        
        if os.path.isfile(drive_path_val):
             fname = os.path.basename(drive_path_val)
             if fname.lower().endswith('.zip'):
                 log(f"Copying and unzipping {fname}...")
                 shutil.copy(drive_path_val, f"./{fname}")
                 run_command(f"unzip -q \"{fname}\"")
             else:
                 log(f"Copying {fname}...")
                 shutil.copy(drive_path_val, ".")
        elif os.path.isdir(drive_path_val):
            log(f"Target is a directory: {drive_path_val}. Using it directly via Source_Path is recommended if it is accessible.")

    elif data_source_val == "Upload Zip":
        log("Checking upload widget...")
        if 'btn_upload' in globals() and btn_upload.value:
            # Handle FileUpload Widget
            uploaded_data = btn_upload.value
            # Support v7 and v8 ipywidgets
            # v7: dict {name: {content: b...}}
            # v8: list/tuple [{name:..., content:b...}]
            
            files_to_process = []
            if isinstance(uploaded_data, dict):
                 for name, info in uploaded_data.items():
                     files_to_process.append({'name': name, 'content': info['content']})
            elif isinstance(uploaded_data, (list, tuple)):
                 for item in uploaded_data:
                     # Item might be a dict (v8) or similar
                     if isinstance(item, dict):
                         files_to_process.append({'name': item.get('name'), 'content': item.get('content')})
                     else:
                         # Fallback for weird objects if any
                         pass
            
            if not files_to_process:
                log("No file content found in widget. Did you upload?")
                return

            for f_info in files_to_process:
                fname = f_info['name']
                content = f_info['content']
                log(f"Processing uploaded file: {fname}")
                with open(fname, 'wb') as f:
                    f.write(content)
                
                if fname.lower().endswith('.zip'):
                    log(f"Unzipping {fname}...")
                    run_command(f"unzip -q \"{fname}\"")
            
            # Clear widget to free memory (optional, specific versions)
            # btn_upload.value.clear() 
            # btn_upload._counter = 0 
        else:
            log("Using legacy files.upload() fallback...")
            try:
                 uploaded = files.upload()
                 for fn in uploaded.keys():
                     log(f"Unzipping {fn}...")
                     run_command(f"unzip -q \"{fn}\"")
            except Exception as e:
                 log(f"Upload failed/cancelled: {e}")

    elif data_source_val == "Custom URL":
        if not custom_url_val:
             log("Error: Custom URL is empty.")
             return
        log(f"Downloading from {custom_url_val}...")
        fname = os.path.basename(custom_url_val)
        if '?' in fname: fname = fname.split('?')[0]
        if not fname: fname = "downloaded_data.zip"

        run_command(f"wget -O {fname} {custom_url_val}")
        
        if fname.lower().endswith('.zip'):
            log(f"Unzipping {fname}...")
            run_command(f"unzip -q \"{fname}\"")
        else:
            log(f"Downloaded {fname}.")
            
    elif data_source_val == "Local Folder":
        if not local_folder_val:
             log("Error: Local Folder path is empty.")
             return
        if not os.path.exists(local_folder_val):
             log(f"Error: Path {local_folder_val} does not exist.")
             return
        log(f"Using Local Folder: {local_folder_val}")

    log("Data preparation complete.")


def start_training(b):
    log("\n=== 4. Starting Training ===")
    os.chdir('/content/gaussian-splatting')
    
    dry_run_val = DryRun
    if 'cb_dryrun' in globals():
        dry_run_val = cb_dryrun.value
    
    cmd_parts = [sys.executable, "train.py"]
    cmd_parts.extend(["-s", Source_Path])
    cmd_parts.extend(["-m", Output_Path])
    cmd_parts.extend(["--iterations", str(Iterations)])
    cmd_parts.extend(["--sh_degree", str(SH_Degree)])
    if White_Background: cmd_parts.append("-w")
    if Eval_Mode: cmd_parts.append("--eval")
    
    # New Features Arguments
    if 'cb_antialiasing' in globals() and cb_antialiasing.value:
        cmd_parts.append("--antialiasing")
    
    if 'cb_exposure' in globals() and cb_exposure.value:
        # Defaults from README
        cmd_parts.extend(["--exposure_lr_init", "0.001", "--exposure_lr_final", "0.0001", "--exposure_lr_delay_steps", "5000", "--exposure_lr_delay_mult", "0.001", "--train_test_exp"])
        
    if 'cb_depth' in globals() and cb_depth.value:
        if 'txt_depth_path' in globals() and txt_depth_path.value:
             cmd_parts.extend(["-d", txt_depth_path.value])
        else:
             log("[WARNING] Depth Regularization enabled but no path provided!")

    if 'cb_sparse_adam' in globals() and cb_sparse_adam.value:
        cmd_parts.extend(["--optimizer_type", "sparse_adam"])

    cmd = " ".join(shlex.quote(arg) for arg in cmd_parts)
    
    if dry_run_val or not torch.cuda.is_available():
        log("--- Dry Run Mode or No GPU ---")
        log(f"Command: {cmd}")
    else:
        log(f"Executing: {cmd}")
        run_command(cmd)


def start_viewer(b):
    log("\n=== 5. Starting Viewer ===")
    
    # Install Cloudflared if needed
    if not os.path.exists('/usr/local/bin/cloudflared') and not os.path.exists('/content/cloudflared-linux-amd64.deb'):
        log("Installing Cloudflared...")
        run_command("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb")
        run_command("dpkg -i /content/cloudflared-linux-amd64.deb")
    
    log("Starting Tunnel and File Server...")
    # Start Tunnel in background
    metrics_port = randint(8100, 9000)
    port = 8000
    
    subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    
    log("Waiting for Cloudflare Tunnel URL...")
    import requests
    tunnel_url = None
    for _ in range(20):
        time.sleep(2)
        try:
            resp = requests.get(f'http://127.0.0.1:{metrics_port}/metrics')
            found = re.search(r"(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", resp.text)
            if found:
                tunnel_url = found.group("url")
                break
        except:
            pass
    
    if tunnel_url:
        log(f"\n>>> Files accessible at: {tunnel_url} <<<\n")
        os.environ['webui_url'] = tunnel_url
    else:
        log("Failed to get Tunnel URL.")

    # Start HTTP Server
    serve_dir = Output_Path if os.path.exists(Output_Path) else "/content/gaussian-splatting"
    if not os.path.exists(serve_dir):
        os.makedirs(serve_dir, exist_ok=True)
    
    log(f"Serving {serve_dir} on port {port}...")
    subprocess.Popen([sys.executable, "-m", "http.server", str(port)], cwd=serve_dir)


def toggle_tailscale(change):
    val = change['new']
    if val:
        log("\n=== Enabling Tailscale ===")
        # Check if installed
        if not os.path.exists('/usr/bin/tailscale'):
             log("Installing Tailscale...")
             run_command("curl -fsSL https://tailscale.com/install.sh | sh")
        
        log("Configuring Hostname: colab")
        run_command("hostname colab")
        
        # Check if daemon running
        # We can just try to start it, if running it might fail or ignore
        log("Starting Tailscale Daemon...")
        run_command("nohup tailscaled --tun=userspace-networking --socket=/run/tailscale/tailscaled.sock --port 41641  >/dev/null 2>&1 &")
        
        log("Connecting...")
        run_command("tailscale up --ssh --hostname=colab")
        log("Tailscale Connected (check output for login link if needed).")
    else:
        log("\n=== Disabling Tailscale ===")
        run_command("tailscale down")
        log("Tailscale Disconnected.")



In [ ]:

# @title 4. Launch Controller
# ===========================
# GUI Construction
# ===========================

style = {'description_width': 'initial'}
layout = widgets.Layout(width='auto')

# --- Widgets ---
cb_tailscale = widgets.Checkbox(value=False, description='Connect Tailscale (ssh)', style=style) # Default False, let user click
cb_dryrun = widgets.Checkbox(value=DryRun, description='Dry Run (No GPU)', style=style)

def on_tailscale_change(change):
    toggle_tailscale(change)
cb_tailscale.observe(on_tailscale_change, names='value')

# Data Input
dd_datasource = widgets.Dropdown(
    options=['Demo Data', 'Google Drive', 'Upload Zip', 'Custom URL', 'Local Folder'],
    value=DataSource,
    description='Data Source:',
    style=style
)
txt_source_path = widgets.Text(
    value=DrivePath if DataSource not in ['Custom URL', 'Local Folder'] else (Custom_URL if DataSource == 'Custom URL' else ''),
    placeholder='Drive Path / URL / Folder Path',
    description='Path / URL:',
    style=style
)
btn_upload = widgets.FileUpload(
    accept='.zip',
    multiple=True,
    description='Upload Zip',
    style=style
)
btn_upload.layout.display = 'none' # Hidden by default

# New Feature Widgets
dd_rasterizer = widgets.Dropdown(
    options=['Standard', 'Accelerated (Sparse Adam)'],
    value='Standard',
    description='Rasterizer:',
    style=style
)

cb_antialiasing = widgets.Checkbox(value=False, description='Anti-aliasing', style=style)
cb_exposure = widgets.Checkbox(value=False, description='Exposure Comp', style=style)
cb_depth = widgets.Checkbox(value=False, description='Depth Reg', style=style)
txt_depth_path = widgets.Text(value='', placeholder='Depth maps path', description='Depth Path:', display='none', style=style)
cb_sparse_adam = widgets.Checkbox(value=False, description='Sparse Adam', disabled=True, style=style) 

# Logic for visibility/interaction
def on_rasterizer_change(change):
    if change['new'].startswith('Accelerated'):
        cb_sparse_adam.value = True
    else:
        cb_sparse_adam.value = False
dd_rasterizer.observe(on_rasterizer_change, names='value')

def on_depth_change(change):
    if change['new']: # Checked
        txt_depth_path.layout.display = 'flex'
    else:
        txt_depth_path.layout.display = 'none'
cb_depth.observe(on_depth_change, names='value')
txt_depth_path.layout.display = 'none' # Init hidden

# --- File Browser Impl ---
current_path = os.getcwd() 
lbl_path = widgets.Label(f"Current: {current_path}")
sel_files = widgets.Select(options=[], rows=10, layout=widgets.Layout(width='100%'))
btn_up = widgets.Button(description="⬆ Up", layout=widgets.Layout(width='80px'))
btn_select = widgets.Button(description="Select", button_style='primary', layout=widgets.Layout(width='80px'))
btn_browse = widgets.Button(description="📂 Browse", layout=widgets.Layout(width='100px'))

# Logic for browser
def update_browser(b=None):
    global current_path
    try:
        items = sorted(os.listdir(current_path))
        formatted_items = []
        for item in items:
            if os.path.isdir(os.path.join(current_path, item)):
                formatted_items.append(f"📁 {item}")
            else:
                formatted_items.append(f"📄 {item}")
        sel_files.options = formatted_items
        lbl_path.value = f"Current: {current_path}"
    except Exception as e:
        lbl_path.value = f"Error: {e}"

def on_up(b):
    global current_path
    current_path = os.path.dirname(current_path)
    update_browser()

def on_select_item(change):
    global current_path
    if change['new']:
        name = change['new'].split(' ', 1)[1]
        full_path = os.path.join(current_path, name)
        if os.path.isdir(full_path):
             current_path = full_path
             update_browser()

def on_confirm_select(b):
    val = sel_files.value
    path_to_use = current_path
    if val:
        name = val.split(' ', 1)[1]
        path_to_use = os.path.join(current_path, name)
    # If browsing for depth?
    txt_source_path.value = path_to_use
    browser_box.layout.display = 'none'

def toggle_browser(b):
    if browser_box.layout.display == 'none':
        browser_box.layout.display = 'block'
        update_browser()
    else:
        browser_box.layout.display = 'none'

btn_up.on_click(on_up)
sel_files.observe(on_select_item, names='value')
btn_select.on_click(on_confirm_select)
btn_browse.on_click(toggle_browser)

browser_box = widgets.VBox([
    widgets.HBox([btn_up, lbl_path]),
    sel_files,
    btn_select
])
browser_box.layout.display = 'none' # Hidden by default

def on_datasource_change(change):
    val = change['new']
    if val in ['Google Drive', 'Local Folder']:

        txt_source_path.layout.display = 'flex'
        btn_browse.layout.display = 'block'
        btn_upload.layout.display = 'none'
    elif val == 'Upload Zip':
        txt_source_path.layout.display = 'none'
        btn_browse.layout.display = 'none'
        btn_upload.layout.display = 'block'
    elif val == 'Custom URL':
        txt_source_path.layout.display = 'flex'
        btn_browse.layout.display = 'none'
        btn_upload.layout.display = 'none'
    else: # Demo Data
        txt_source_path.layout.display = 'none'
        btn_browse.layout.display = 'none'
        btn_upload.layout.display = 'none'

dd_datasource.observe(on_datasource_change, names='value')


# --- Buttons ---
btn_env = widgets.Button(description="1. Setup Environment", button_style='primary', layout=layout)
btn_env.on_click(setup_env)

btn_deps = widgets.Button(description="2. Install Dependencies", button_style='info', layout=layout)
btn_deps.on_click(install_deps)

btn_data = widgets.Button(description="3. Prepare Data", button_style='warning', layout=layout)
btn_data.on_click(prepare_data)

btn_train = widgets.Button(description="4. Train", button_style='success', layout=layout)
btn_train.on_click(start_training)

btn_view = widgets.Button(description="5. Start Viewer", button_style='danger', layout=layout)
btn_view.on_click(start_viewer)

# --- Layout ---
gui = widgets.VBox([
    widgets.HTML("<h3>Gaussian Splatting Controller</h3>"),
    widgets.HBox([cb_tailscale]),
    widgets.HBox([btn_env]),
    widgets.HBox([dd_rasterizer, btn_deps]),
    widgets.HTML("<hr>"),
    widgets.HBox([dd_datasource, txt_source_path, btn_browse]),
    widgets.HBox([btn_upload]),
    browser_box,
    widgets.HBox([btn_data]),
    widgets.HTML("<hr>"),
    widgets.Label("Training Options (New Features):"),
    widgets.HBox([cb_antialiasing, cb_exposure, cb_sparse_adam]),
    widgets.HBox([cb_depth, txt_depth_path]),
    widgets.HTML("<br>"),
    widgets.HBox([cb_dryrun]),
    widgets.HBox([btn_train, btn_view]),
    widgets.Label("Logs:"),
    out
])

# Trigger initial state
on_datasource_change({'new': dd_datasource.value})

display(gui)

